# Phase 1: Self-Supervised Pre-Training (70% Subject Split)

This notebook implements the first phase of the experimental pipeline proposed by Corponi et al.
It clones the source repository, integrates the pre-processed dataset, and trains the
time-series Transformer using a masked-prediction pretext task on 70% of the available subjects.
The remaining 30% are held out and reserved exclusively for Phase 2 (supervised fine-tuning),
ensuring strict subject-level data separation and preventing data leakage.

### 1. Repository Cloning
Cloning the e4selflearning source repository. Dependency installation is performed later to ensure
the native NumPy version is active during dataset loading.

In [ ]:
import os, sys, shutil

WORK_DIR = '/kaggle/working/e4selflearning'
os.chdir('/kaggle/working')

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

print("Cloning repository...")
!git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}
print("Repository ready.")

### 2. Dataset Integration
Locating the pre-processed dataset from the Kaggle Input directory and creating symbolic links.
This step runs with the native environment NumPy (2.x) to ensure correct deserialization
of the pre-existing metadata.pkl binary file.

In [ ]:
import os, glob, shutil, pickle

TARGET_DIR = '/kaggle/working/e4selflearning/data/preprocessed/unsegmented_unlabelled'
if os.path.exists(TARGET_DIR): shutil.rmtree(TARGET_DIR)
os.makedirs(TARGET_DIR, exist_ok=True)

print("Searching for dataset in Kaggle Input directory...")
meta_files = glob.glob('/kaggle/input/**/metadata.pkl', recursive=True)
if not meta_files:
    raise FileNotFoundError("Error: metadata.pkl not found. Verify the pre-processed dataset has been added.")
src_dir = os.path.dirname(meta_files[0])
print(f"Source path: {src_dir}")

for item in os.listdir(src_dir):
    s, d = os.path.join(src_dir, item), os.path.join(TARGET_DIR, item)
    if item == 'metadata.pkl':
        shutil.copy2(s, d)
    else:
        os.symlink(s, d)

meta_p = os.path.join(TARGET_DIR, 'metadata.pkl')
with open(meta_p, 'rb') as f: meta = pickle.load(f)

new_sessions = {}
found_count = 0
for root, dirs, files in os.walk(TARGET_DIR, followlinks=True):
    if 'channels.h5' in files:
        clean_name = os.path.basename(root)
        new_rel_id = os.path.relpath(root, TARGET_DIR).replace('\\', '/')
        for old_id in meta['sessions_info'].keys():
            if old_id.endswith(clean_name):
                new_sessions[new_rel_id] = meta['sessions_info'][old_id]
                found_count += 1
                break

if found_count > 0:
    meta['sessions_info'] = new_sessions
    with open(meta_p, 'wb') as f: pickle.dump(meta, f)
    print(f"Loading complete. Mapped sessions: {found_count}")

### 3. Dependency Installation
Installing the required scientific computing stack with pinned versions for reproducibility.
NumPy is downgraded to 1.26.4 for compatibility with the Corponi model utilities.

In [ ]:
print("Installing dependencies...")
!pip uninstall -y -q numpy scipy pandas
!pip install --no-cache-dir -q "numpy==1.26.4" "scipy==1.12.0" "pandas<2.2" biopsykit flirt ruamel.yaml biosppy hrv-analysis neurokit2 distinctipy joypy peakutils lightning torchmetrics
print("Dependencies installed.")

### 4. Codebase Patching
Applying minimal source-level fixes to ensure compatibility with modern Python environments and custom metadata structures.

In [ ]:
import os
os.chdir('/kaggle/working/e4selflearning')

r_p = 'src/timebase/data/reader.py'
if os.path.exists(r_p):
    with open(r_p, 'r') as f: c = f.read()
    c = c.replace('idx_dict_cases["train"] + idx_dict_controls["train"])', 'idx_dict_cases["train"] + idx_dict_controls["train"], dtype=np.int64)')
    c = c.replace('idx_dict_cases["val"] + idx_dict_controls["val"])', 'idx_dict_cases["val"] + idx_dict_controls["val"], dtype=np.int64)')
    c = c.replace('idx_dict_cases["test"] + idx_dict_controls["test"])', 'idx_dict_cases["test"] + idx_dict_controls["test"], dtype=np.int64)')
    with open(r_p, 'w') as f: f.write(c)

!sed -i 's/v.to(device)/v.float().to(device)/g' pre_train.py

h5_p = 'src/timebase/utils/h5.py'
if os.path.exists(h5_p):
    with open(h5_p, 'r') as f: c = f.read()
    c = c.replace('data=v,', 'data=v, compression="gzip", compression_opts=4,')
    with open(h5_p, 'w') as f: f.write(c)

static_p = 'src/timebase/data/static.py'
if os.path.exists(static_p):
    with open(static_p, 'r') as f: c = f.read()
    if '"dati_preelaborati"' not in c:
        c = c.replace('"wesd": 12,', '"wesd": 12,\n    "dati_preelaborati": 0,')
        with open(static_p, 'w') as f: f.write(c)

print("Patches applied successfully.")

### 5. Neural Segmentation
Slicing continuous time-series recordings into fixed-length windows (length=512, step=256) using the sliding-window segmentation utility.

In [ ]:
import os
os.chdir('/kaggle/working/e4selflearning')
BASE = 'data/preprocessed/unsegmented_unlabelled'
SEG_DIR = 'data/preprocessed/sl512_ss256_unlabelled'
print("Starting segmentation process...")
!PYTHONPATH=src python segment.py --data_dir {BASE} --output_dir {SEG_DIR} --segment_length 512 --segmentation_mode 1 --step_size 256 --num_workers 2 --overwrite
print("Segmentation complete.")

### 6. Subject-Level Split (70% Training Partition)
Isolating 70% of the unique subjects for pre-training. The split is deterministic (seed=42) so that Phase 2 can reconstruct the complementary 30% partition exactly.

In [ ]:
import pickle, os, numpy as np, re

SEG_DIR = '/kaggle/working/e4selflearning/data/preprocessed/sl512_ss256_unlabelled'
meta_path = os.path.join(SEG_DIR, "metadata.pkl")
with open(meta_path, "rb") as f: meta = pickle.load(f)

all_paths = meta["sessions_paths"]
if "Sub_ID" not in meta.get("sessions_labels", {}):
    meta.setdefault("sessions_labels", {})["Sub_ID"] = [np.nan] * len(all_paths)

sub_ids = []
for path in all_paths:
    p_norm = path.replace('\\', '/').replace('//', '/')
    parts = p_norm.split('/')
    final_sid = None
    for pt in parts:
        if 'sl512' in pt or 'ss256' in pt: continue
        match = re.search(r'[sS]ubject_?(\d+)|[sS](\d+)', pt)
        if match:
            val = match.group(1) or match.group(2)
            final_sid = f"{parts[0][:3].lower()}_{val}"
            break
    if not final_sid and len(parts) >= 2:
        final_sid = f"folder_{parts[-2]}"
    sub_ids.append(final_sid)

unique_subs = np.unique(sub_ids)
unique_subs.sort()
np.random.seed(42)
np.random.shuffle(unique_subs)

split_idx = int(len(unique_subs) * 0.70)
fase1_subs = set(unique_subs[:split_idx])

filtered_paths, filtered_sub_ids = [], []
for p, s in zip(all_paths, sub_ids):
    if s in fase1_subs:
        filtered_paths.append(p)
        filtered_sub_ids.append(s)

meta["sessions_paths"] = np.array(filtered_paths)
meta["sessions_labels"] = {"Sub_ID": np.array(filtered_sub_ids)}
with open(meta_path, "wb") as f: pickle.dump(meta, f)

print(f"Subject-level split applied.")
print(f"Total unique subjects: {len(unique_subs)}")
print(f"Phase 1 partition (70%): {len(fase1_subs)} subjects, {len(filtered_paths)} segments.")

### 7. SSL Pre-Training Execution
Running the masked-prediction pretext task on the isolated 70% partition.

In [ ]:
import os
os.chdir('/kaggle/working/e4selflearning')
EPOCHS = 50
OUTPUT_DIR = 'output_pretrain'
SEG_DATA = 'data/preprocessed/sl512_ss256_unlabelled'
print("Starting Self-Supervised Pre-Training...")
!PYTHONPATH=src python pre_train.py --dataset {SEG_DATA} --pretext_task masked_prediction --output_dir {OUTPUT_DIR} --epochs {EPOCHS} --batch_size 32 --e4selflearning --verbose 1
print(f"Training complete. Model checkpoints saved in: {OUTPUT_DIR}")

### 8. Results Archiving
Packaging the trained model checkpoints into a compressed archive for use in Phase 2.

In [ ]:
import shutil
shutil.make_archive('trained_model_fase1', 'gztar', '/kaggle/working/e4selflearning/output_pretrain')
print("Model archived. Download 'trained_model_fase1.tar.gz' from the Output tab.")